In [14]:
import sys
import maboss
import pandas as pd
import numpy as np
sys.path.append("/Users/emilieyu/endotehelial-masboss")
import matplotlib.pyplot as plt

In [2]:
from src.config import load_sim_config, load_sweep_config, load_spatial_config
from src.paths import *
from boolean_models.scripts import run_perturbations, run_sweeps

sim_cfg = load_sim_config()
sweep_cfg = load_sweep_config()
spatial_cfg = load_spatial_config()

MODELS_BND = DATA_DIR / sim_cfg['model']['bnd']
MODELS_CFG = DATA_DIR / sim_cfg['model']['cfg_v3']

base_model = maboss.load(str(MODELS_BND), str(MODELS_CFG))
base_model.param['max_time'] = 10.0
base_model.param['sample_count'] = 5000
spatial_cfg['hill_params']['JCAD']
MODELS_CFG

PosixPath('/Users/emilieyu/endotehelial-masboss/data/models/rho_v3.cfg')

In [10]:
from abm.flow_field import FlowField 
from abm.endothelial_cell import EndothelialCell
from abm.membrane_node import MembraneNode

## Sigmoid Parameter Activation

In [20]:
cell = EndothelialCell(cell_id=1, centroid=np.array([1.1, 0.0]))
cell.positions

# plot cell (exploratory)
# plt.scatter(cell.positions[:,0], cell.positions[:,1], s=10)
# plt.show()
# plt.close()

array([[ 1.11000000e+01,  0.00000000e+00],
       [ 9.76025404e+00,  5.00000000e+00],
       [ 6.10000000e+00,  8.66025404e+00],
       [ 1.10000000e+00,  1.00000000e+01],
       [-3.90000000e+00,  8.66025404e+00],
       [-7.56025404e+00,  5.00000000e+00],
       [-8.90000000e+00,  1.22464680e-15],
       [-7.56025404e+00, -5.00000000e+00],
       [-3.90000000e+00, -8.66025404e+00],
       [ 1.10000000e+00, -1.00000000e+01],
       [ 6.10000000e+00, -8.66025404e+00],
       [ 9.76025404e+00, -5.00000000e+00]])

In [22]:
field = FlowField()
field.classify_faces(cell.positions)

ValueError: either both or neither of x and y should be given

In [ ]:
def hill(tau, K, n):
    """
    S: Sensitivity input (sheae nmagnitude / force)
    K: Half activation thershold
    n: Hill coefficiet
    """
    return tau**n / (K**n + tau**n)

def get_protein_activation(cfg, tau, face_type, protein):
    hill_params = cfg['hill_params'][protein]
    face_biases = cfg['face_bias'][protein]

    K = hill_params['K']
    n = hill_params['n']
    bias = face_biases[face_type]

    # Effective shear felt by protein at specified face
    tau_eff = tau * bias

    return hill (tau_eff, K ,n)

def shear_to_kinetic(cfg, tau, face_type):
    """
    Hill activation probabilities sclae MaBoSS kinetic rates.

    Protein signalling efficiency varies continuously with shear.
    Face type modulates effective recruitment. 
    """
    rho_params = {
        "$RhoA_amp": 10.0,
        "$RhoA_mod": 1.2,
        "$RhoA_basal": 0.5,
        "$RhoA_antagonistic": 3.0,
        "$RhoA_decay": 0.2,
        
        "$RhoC_amp": 10.0,
        "$RhoC_mod": 1.2,
        "$RhoC_basal": 0.5,
        "$RhoC_antagonistic": 3.0,
        "$RhoC_decay": 0.2
    }
    
    p_dsp = get_protein_activation(tau, face_type, 'DSP')
    p_tjp1 = get_protein_activation(tau, face_type, 'TJP1')
    p_jcad = get_protein_activation(tau, face_type, 'JCAD')

    rho_params['$RhoA_amp'] = rho_params['$RhoA_amp'] * p_dsp * p_jcad
    rho_params['$RhoA_mod'] = rho_params['$RhoA_mod'] * p_dsp * (1-p_jcad)

    rho_params['$RhoC_amp'] = rho_params['$RhoC_amp'] * p_tjp1 * p_jcad
    rho_params['$RhoC_mod'] = rho_params['$RhoC_mod'] * p_tjp1 * (1-p_jcad)


    return rho_params

In [9]:
get_protein_activation(spatial_cfg, 0.4, 'downstream', 'DSP')

0.017467248908296953

centre: [2. 0.]
relative: [[-2.   0. ]
 [-1.   0.8]
 [-1.  -0.8]
 [ 0.   1.5]
 [ 0.  -1.5]
 [ 1.   1. ]
 [ 1.  -1. ]
 [ 2.   0. ]]
norms: [[2.        ]
 [1.28062485]
 [1.28062485]
 [1.5       ]
 [1.5       ]
 [1.41421356]
 [1.41421356]
 [2.        ]]
outward_normals [[-1.          0.        ]
 [-0.78086881  0.62469505]
 [-0.78086881 -0.62469505]
 [ 0.          1.        ]
 [ 0.         -1.        ]
 [ 0.70710678  0.70710678]
 [ 0.70710678 -0.70710678]
 [ 1.          0.        ]]
alignment [-1.         -0.78086881 -0.78086881  0.          0.          0.70710678
  0.70710678  1.        ]


array(['upstream', 'upstream', 'upstream', 'lateral', 'lateral',
       'downstream', 'downstream', 'downstream'], dtype='<U10')

magnitude: -0.9999999999333333
force_vector [-1. -0.]


## Testing DataStructures

In [53]:
positions = np.ndarray(2)
positions

array([-0., -1.])

## Run Perturbation and Get Result

In [3]:
perb_df, ss_df = run_perturbations(base_model, sim_cfg)

DEBUG: Running perturbation: WT
DEBUG: Running perturbation: DSP_KO
DEBUG: Running perturbation: TJP1_KO
DEBUG: Running perturbation: JCAD_KO
DEBUG: Running perturbation: DSP_JCAD_DKO
DEBUG: Running perturbation: TJP1_JCAD_DKO
DEBUG: All simulations completed successfully


In [4]:
ss_df

,t,DSP,TJP1,JCAD,RhoA,RhoC,perturbation,delta,phenotype
0,9.9,0.000000,0.498400,0.000000,0.395196,0.558345,DSP_JCAD_DKO,0.163149,Normal
1,9.9,0.000000,0.492999,0.491800,0.353242,0.627831,DSP_KO,0.274589,Hyper
2,9.9,0.506201,0.485401,0.000000,0.506329,0.501611,JCAD_KO,-0.004718,Normal
3,9.9,0.503400,0.000000,0.000000,0.566635,0.394151,TJP1_JCAD_DKO,-0.172484,Normal
4,9.9,0.496400,0.000000,0.496600,0.634992,0.353916,TJP1_KO,-0.281076,Failed
5,9.9,0.510399,0.500000,0.517999,0.542614,0.552133,WT,0.009519,Normal


In [ ]:
def sigmoid(x, threshold, steepness=5.0):
    return 1 / (1.0 + np.exp(-steepness * (x - threshold)))